# 04 - SQL Analysis

## Introduction

This notebook transforms the cleaned incident dataset into a local SQLite analytics layer and then runs reusable SQL queries for reporting and Power BI-ready exports.

Notebook 03 remains dedicated to pandas-based exploratory data analysis and visualizations, while Notebook 04 is dedicated to SQL analysis and business reporting.

Workflow:
Clean CSV -> SQLite -> SQL queries -> pandas results -> CSV exports -> Power BI

## Executive Summary

This notebook builds a reusable SQLite analytics layer from the cleaned incident dataset.

The SQL queries focus on:
- Operational KPIs
- SLA monitoring
- Support team performance
- Resolution bottlenecks
- Power BI data exports

The exported datasets are used to power the interactive dashboard developed in the final stage of the project.

## Workflow

```text
incident_clean.csv
        │
        ▼
+------------------------+
| SQLite Database        |
| Table: incidents       |
+------------------------+
        │
        ▼
SQL Analytics
        │
        ▼
Business KPI Tables
        │
        ▼
CSV Exports
        │
        ▼
Power BI Dashboard
```

## Contents

1. [Introduction](#introduction)
2. [Executive Summary](#executive-summary)
3. [Workflow](#workflow)
4. [Load Libraries](#load-libraries)
5. [Load Clean Dataset](#load-clean-dataset)
6. [Build SQLite Analytics Database](#build-sqlite-analytics-database)
7. [Database Validation](#database-validation)
8. [SQL KPI Analysis](#sql-kpi-analysis)
9. [SQL Operational Analysis](#sql-operational-analysis)
10. [Assignment Group Performance](#assignment-group-performance)
11. [Category Risk And Temporal Load](#category-risk-and-temporal-load)
12. [Temporal Analysis](#temporal-analysis)
13. [Resolution Bottlenecks](#resolution-bottlenecks)
14. [Reassignments And Reopens](#reassignments-and-reopens)
15. [SLA Segmentation](#sla-segmentation)
16. [Power BI Exports And Summary](#power-bi-exports-and-summary)
17. [Key SQL Deliverables](#key-sql-deliverables)
18. [Power BI Integration](#power-bi-integration)
19. [Conclusion](#conclusion)

## Load Libraries

Import the Python libraries used to prepare paths, read the cleaned dataset, and display tabular outputs.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)

PROJECT_ROOT = Path('..')
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'incident_clean.csv'
DB_DIR = PROJECT_ROOT / 'data' / 'database'
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / 'incident.db'
EXPORT_DIR = PROJECT_ROOT / 'data' / 'processed'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/incident_clean.csv')
    DB_DIR = Path('data/database')
    DB_DIR.mkdir(parents=True, exist_ok=True)
    DB_PATH = DB_DIR / 'incident.db'
    EXPORT_DIR = Path('data/processed')
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## Load Clean Dataset

The cleaned incident CSV is loaded into pandas with explicit date parsing and a string vendor dtype to keep the dataframe stable for downstream SQL work.

In [2]:
df = pd.read_csv(
    DATA_PATH,
    parse_dates=['opened_at', 'resolved_at', 'closed_at'],
    dtype={'vendor': 'string'},
    low_memory=False,
)

display({'dataset_path': str(DATA_PATH), 'shape': df.shape})

{'dataset_path': '..\\data\\processed\\incident_clean.csv',
 'shape': (24918, 58)}

## Build SQLite Analytics Database

The cleaned dataframe is written to a local SQLite database so the SQL analysis can reuse a persistent, queryable table.

In [3]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
df.to_sql('incidents', conn, if_exists='replace', index=False)

display({
    'SQLite database': str(DB_PATH),
    'Table': 'incidents',
    'Rows imported': len(df)
})

def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

def save_sql_result(query: str, output_name: str) -> pd.DataFrame:
    result = run_sql(query)
    result.to_csv(EXPORT_DIR / output_name, index=False)
    return result

{'SQLite database': '..\\data\\database\\incident.db',
 'Table': 'incidents',
 'Rows imported': 24918}

## Database Validation

These checks confirm the row count, the main dimensions, and the SQLite schema before the KPI analysis starts.

In [4]:
schema_overview = run_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT number) AS distinct_incidents,
    COUNT(DISTINCT assignment_group) AS assignment_groups,
    COUNT(DISTINCT category) AS categories,
    COUNT(DISTINCT priority_text) AS priorities
FROM incidents;
""")

table_columns = run_sql("""
PRAGMA table_info(incidents);
""")

display(schema_overview)
display(table_columns)

,total_rows,distinct_incidents,assignment_groups,categories,priorities
0,24918,24918,70,52,4


,cid,name,type,notnull,dflt_value,pk
0,0,number,TEXT,0,None,0
1,1,incident_state,TEXT,0,None,0
2,2,active,INTEGER,0,None,0
3,3,reassignment_count,INTEGER,0,None,0
4,4,reopen_count,INTEGER,0,None,0
5,5,sys_mod_count,INTEGER,0,None,0
6,6,made_sla,INTEGER,0,None,0
7,7,caller_id,TEXT,0,None,0
8,8,opened_by,TEXT,0,None,0
9,9,opened_at,TIMESTAMP,0,None,0


## SQL KPI Analysis

**Business Question:** What is the overall service performance level?

In [5]:
kpi_summary = run_sql("""
SELECT
    COUNT(*) AS total_incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(closure_time_hours), 2) AS avg_closure_hours,
    ROUND(AVG(reassignment_count), 2) AS avg_reassignments,
    ROUND(AVG(reopen_count), 2) AS avg_reopens
FROM incidents;
""")

display(kpi_summary)

,total_incidents,sla_rate_pct,avg_resolution_hours,avg_closure_hours,avg_reassignments,avg_reopens
0,24918,63.44,178.17,315.33,0.94,0.01


## SQL Operational Analysis

**Business Question:** Which operational drivers explain the KPI results?

In [6]:
run_sql("""
SELECT
    priority_text,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(reopen_count), 2) AS avg_reopens
FROM incidents
GROUP BY priority_text
ORDER BY incidents DESC;
""")

,priority_text,incidents,sla_rate_pct,avg_resolution_hours,avg_reopens
0,Moderate,23466,64.55,174.35,0.01
1,Low,774,84.11,283.19,0.00
2,High,408,0.74,152.46,0.01
3,Critical,270,2.22,266.08,0.00


### Interpretation
Priority distribution shows where the incident load is concentrated and how SLA performance varies by severity level.

## Assignment Group Performance

**Business Question:** Which groups process the most incidents, and what SLA and resolution patterns do they show?

In [7]:
run_sql("""
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(reassignment_count), 2) AS avg_reassignments
FROM incidents
WHERE assignment_group IS NOT NULL AND assignment_group <> ''
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY incidents DESC
LIMIT 15;
""")

,assignment_group,incidents,sla_rate_pct,avg_resolution_hours,avg_reassignments
0,Group 70,9444,83.84,50.22,0.50
1,Group 25,1243,42.80,251.94,1.09
2,Group 39,1199,68.81,71.96,1.37
3,Group 24,1060,64.81,136.59,1.03
4,Group 23,811,58.69,245.22,1.58
5,Group 64,716,89.11,18.75,0.16
6,Group 73,576,53.99,124.71,1.02
7,Group 28,545,53.58,135.38,1.27
8,Group 27,518,58.30,123.84,0.99
9,Group 20,394,42.13,230.13,1.99


### Interpretation
Assignment groups with high volume and lower SLA compliance should be prioritized for operational review and staffing adjustments.

## Category Risk And Temporal Load

**Business Question:** Which categories and periods drive SLA pressure?

In [8]:
top_category_sla = run_sql("""
SELECT
    category,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY sla_rate_pct ASC, incidents DESC
LIMIT 10;
""")

monthly_load = run_sql("""
SELECT
    strftime('%Y-%m', opened_at) AS opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY strftime('%Y-%m', opened_at)
ORDER BY opened_month;
""")

hourly_load = run_sql("""
SELECT
    CAST(strftime('%H', opened_at) AS INTEGER) AS opened_hour,
    COUNT(*) AS incidents
FROM incidents
GROUP BY CAST(strftime('%H', opened_at) AS INTEGER)
ORDER BY opened_hour;
""")

display(top_category_sla)
display(monthly_load)
display(hourly_load)

,category,incidents,sla_rate_pct,avg_resolution_hours
0,Category 55,106,20.75,674.02
1,Category 45,602,25.91,481.75
2,Category 34,501,33.53,1326.32
3,Category 40,436,38.99,152.81
4,Category 19,245,46.94,277.65
5,Category 61,810,47.04,182.37
6,Category 46,2432,48.52,316.02
7,Category 44,302,50.00,153.81
8,Category 23,1063,52.49,151.58
9,Category 24,640,52.66,279.68


,opened_month,incidents,sla_rate_pct
0,2016-02,207,64.73
1,2016-03,8995,39.88
2,2016-04,7934,76.70
3,2016-05,7508,77.56
4,2016-06,5,80.00
5,2016-07,14,78.57
6,2016-08,15,46.67
7,2016-09,12,58.33
8,2016-10,16,43.75
9,2016-11,26,65.38


,opened_hour,incidents
0,0,269
1,1,222
2,2,149
3,3,123
4,4,128
5,5,132
6,6,320
7,7,573
8,8,2547
9,9,3014


## Temporal Analysis

**Business Question:** When do incidents arrive, and how does SLA performance vary across time?

In [9]:
monthly_load = run_sql("""
SELECT
    strftime('%Y-%m', opened_at) AS opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY strftime('%Y-%m', opened_at)
ORDER BY opened_month;
""")

weekday_load = run_sql("""
SELECT
    CASE strftime('%w', opened_at)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS weekday,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY strftime('%w', opened_at)
ORDER BY CAST(strftime('%w', opened_at) AS INTEGER);
""")

hourly_load = run_sql("""
SELECT
    CAST(strftime('%H', opened_at) AS INTEGER) AS opened_hour,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY CAST(strftime('%H', opened_at) AS INTEGER)
ORDER BY opened_hour;
""")

business_hours_load = run_sql("""
SELECT
    CASE WHEN is_business_hours = 1 THEN 'Business hours' ELSE 'Outside business hours' END AS period,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY is_business_hours
ORDER BY incidents DESC;
""")

display(monthly_load)
display(weekday_load)
display(hourly_load)
display(business_hours_load)

,opened_month,incidents,sla_rate_pct
0,2016-02,207,64.73
1,2016-03,8995,39.88
2,2016-04,7934,76.70
3,2016-05,7508,77.56
4,2016-06,5,80.00
5,2016-07,14,78.57
6,2016-08,15,46.67
7,2016-09,12,58.33
8,2016-10,16,43.75
9,2016-11,26,65.38


,weekday,incidents,sla_rate_pct
0,Sunday,563,72.29
1,Monday,5947,64.47
2,Tuesday,4951,61.66
3,Wednesday,4884,59.58
4,Thursday,4105,62.27
5,Friday,3829,66.44
6,Saturday,639,78.87


,opened_hour,incidents,sla_rate_pct
0,0,269,72.49
1,1,222,69.82
2,2,149,78.52
3,3,123,62.60
4,4,128,72.66
5,5,132,68.18
6,6,320,76.25
7,7,573,69.28
8,8,2547,69.41
9,9,3014,66.06


,period,incidents,sla_rate_pct,avg_resolution_hours
0,Business hours,18156,60.57,204.37
1,Outside business hours,6762,71.15,106.73


## Resolution Bottlenecks

**Business Question:** Which incidents, categories, and groups take the longest to resolve?

In [10]:
top_long_incidents = run_sql("""
SELECT
    number,
    assignment_group,
    category,
    priority_text,
    resolution_time_hours,
    closure_time_hours
FROM incidents
WHERE resolution_time_hours IS NOT NULL
ORDER BY resolution_time_hours DESC
LIMIT 10;
""")

slow_categories = run_sql("""
SELECT
    category,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY avg_resolution_hours DESC, incidents DESC
LIMIT 10;
""")

slow_groups = run_sql("""
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY avg_resolution_hours DESC, incidents DESC
LIMIT 10;
""")

resolution_summary = run_sql("""
SELECT
    COUNT(*) AS incidents_with_resolution_time,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(MIN(resolution_time_hours), 2) AS min_resolution_hours,
    ROUND(MAX(resolution_time_hours), 2) AS max_resolution_hours
FROM incidents
WHERE resolution_time_hours IS NOT NULL;
""")

display(top_long_incidents)
display(slow_categories)
display(slow_groups)
display(resolution_summary)

,number,assignment_group,category,priority_text,resolution_time_hours,closure_time_hours
0,INC0001839,Group 31,Category 45,Moderate,8070.17,8190.62
1,INC0001349,Group 67,Category 45,Low,8020.27,8140.78
2,INC0007343,Group 31,Category 46,Moderate,7803.68,7923.73
3,INC0001881,Group 35,Category 55,Low,7316.45,7436.83
4,INC0001978,Group 35,Category 55,Low,7313.93,7434.30
5,INC0001984,Group 35,Category 55,Low,7313.87,7434.22
6,INC0019986,Group 3,Category 42,Moderate,7220.58,7341.28
7,INC0000343,Group 31,Category 45,Low,7122.28,7242.77
8,INC0000307,Group 66,Category 45,Low,7056.42,7176.50
9,INC0003982,Group 47,Category 37,Moderate,6729.00,6849.10


,category,incidents,avg_resolution_hours,sla_rate_pct
0,Category 34,501,1326.32,33.53
1,Category 55,106,674.02,20.75
2,Category 45,602,481.75,25.91
3,Category 46,2432,316.02,48.52
4,Category 24,640,279.68,52.66
5,Category 19,245,277.65,46.94
6,Category 61,810,182.37,47.04
7,Category 57,971,161.84,59.84
8,Category 9,1155,156.23,59.22
9,Category 44,302,153.81,50.00


,assignment_group,incidents,avg_resolution_hours,sla_rate_pct
0,Group 31,205,758.30,32.20
1,Group 10,343,597.16,25.07
2,Group 66,376,498.59,30.32
3,Group 37,176,412.19,25.00
4,Group 33,201,406.82,38.81
5,Group 12,123,335.77,13.82
6,Group 57,313,312.57,31.95
7,Group 72,367,311.16,36.78
8,Group 6,257,305.25,44.36
9,Group 25,1243,251.94,42.80


,incidents_with_resolution_time,avg_resolution_hours,min_resolution_hours,max_resolution_hours
0,23362,178.17,0.0,8070.17


## Reassignments And Reopens

**Business Question:** Do reassignments and reopens correlate with slower delivery?

In [11]:
reassign_distribution = run_sql("""
SELECT
    reassignment_count,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY reassignment_count
ORDER BY reassignment_count;
""")

reopen_distribution = run_sql("""
SELECT
    reopen_count,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY reopen_count
ORDER BY reopen_count;
""")

reassignment_bucket = run_sql("""
SELECT
    CASE
        WHEN reassignment_count = 0 THEN '0'
        WHEN reassignment_count = 1 THEN '1'
        WHEN reassignment_count BETWEEN 2 AND 3 THEN '2-3'
        ELSE '4+'
    END AS bucket,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY bucket
ORDER BY incidents DESC;
""")

reopen_bucket = run_sql("""
SELECT
    CASE
        WHEN reopen_count = 0 THEN '0'
        WHEN reopen_count = 1 THEN '1'
        WHEN reopen_count BETWEEN 2 AND 3 THEN '2-3'
        ELSE '4+'
    END AS bucket,
    COUNT(*) AS incidents,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY bucket
ORDER BY incidents DESC;
""")

display(reassign_distribution)
display(reopen_distribution)
display(reassignment_bucket)
display(reopen_bucket)

,reassignment_count,incidents,avg_resolution_hours
0,0,13549,94.98
1,1,6238,173.32
2,2,2283,315.63
3,3,1234,372.60
4,4,695,418.61
5,5,381,449.43
6,6,183,580.11
7,7,143,485.91
8,8,70,521.49
9,9,52,644.98


,reopen_count,incidents,avg_resolution_hours
0,0,24643,174.86
1,1,245,452.30
2,2,19,364.90
3,3,4,313.97
4,4,4,1039.83
5,6,2,968.46
6,8,1,675.15


,bucket,incidents,avg_resolution_hours,sla_rate_pct
0,0,13549,94.98,78.39
1,1,6238,173.32,57.09
2,2-3,3517,335.62,37.48
3,4+,1614,472.47,19.08


,bucket,incidents,avg_resolution_hours,sla_rate_pct
0,0,24643,174.86,63.88
1,1,245,452.30,26.53
2,2-3,23,356.05,4.35
3,4+,7,967.34,14.29


## SLA Segmentation

**Business Question:** Which priorities, impacts, and urgencies create the highest SLA risk?

In [12]:
sla_by_impact = run_sql("""
SELECT
    impact_text,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY impact_text
ORDER BY sla_rate_pct ASC, incidents DESC;
""")

sla_by_urgency = run_sql("""
SELECT
    urgency_text,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY urgency_text
ORDER BY sla_rate_pct ASC, incidents DESC;
""")

priority_category_risk = run_sql("""
SELECT
    priority_text,
    category,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY priority_text, category
HAVING COUNT(*) >= 100
ORDER BY sla_rate_pct ASC, incidents DESC
LIMIT 15;
""")

sla_by_group = run_sql("""
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY sla_rate_pct ASC, incidents DESC
LIMIT 15;
""")

display(sla_by_impact)
display(sla_by_urgency)
display(priority_category_risk)
display(sla_by_group)

,impact_text,incidents,sla_rate_pct,avg_resolution_hours
0,High,421,1.66,216.30
1,Medium,23746,63.83,174.97
2,Low,751,85.62,262.97


,urgency_text,incidents,sla_rate_pct,avg_resolution_hours
0,High,531,2.07,218.54
1,Medium,23689,64.20,174.61
2,Low,698,84.24,273.14


,priority_text,category,incidents,sla_rate_pct
0,Moderate,Category 45,510,24.90
1,Moderate,Category 34,492,33.33
2,Moderate,Category 40,419,39.14
3,Moderate,Category 61,787,47.78
4,Moderate,Category 44,293,50.17
5,Moderate,Category 46,2242,51.61
6,Moderate,Category 19,210,52.38
7,Moderate,Category 24,619,52.50
8,Moderate,Category 23,977,55.89
9,Moderate,Category 9,1115,58.65


,assignment_group,incidents,sla_rate_pct,avg_resolution_hours
0,Group 12,123,13.82,335.77
1,Group 37,176,25.00,412.19
2,Group 10,343,25.07,597.16
3,Group 66,376,30.32,498.59
4,Group 74,105,30.48,249.77
5,Group 57,313,31.95,312.57
6,Group 31,205,32.20,758.30
7,Group 72,367,36.78,311.16
8,Group 29,257,37.74,205.56
9,Group 33,201,38.81,406.82


#### SQL Insight

Several assignment groups exhibit lower SLA compliance despite comparable incident volumes.

#### Business Value

These results can be used to prioritize operational improvements and staffing decisions.

## Power BI Exports And Summary

**Business Question:** Which SQL outputs should be exported for reporting and dashboards?

In [13]:
export_monthly = save_sql_result("""
SELECT
    strftime('%Y-%m', opened_at) AS opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY strftime('%Y-%m', opened_at)
ORDER BY opened_month;
""", 'sql_monthly_load.csv')

export_priority = save_sql_result("""
SELECT
    priority_text,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY priority_text
ORDER BY incidents DESC;
""", 'sql_priority_sla.csv')

export_groups = save_sql_result("""
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
WHERE assignment_group IS NOT NULL AND assignment_group <> ''
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY incidents DESC
LIMIT 20;
""", 'sql_assignment_group_kpi.csv')

export_top_long = save_sql_result("""
SELECT
    number,
    assignment_group,
    category,
    priority_text,
    resolution_time_hours
FROM incidents
WHERE resolution_time_hours IS NOT NULL
ORDER BY resolution_time_hours DESC
LIMIT 25;
""", 'sql_top_long_incidents.csv')

summary_export = run_sql("""
SELECT
    COUNT(*) AS exported_tables,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS baseline_sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS baseline_resolution_hours
FROM incidents;
""")

export_kpi_summary = save_sql_result("""
SELECT
    COUNT(*) AS total_incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(reassignment_count), 2) AS avg_reassignments,
    ROUND(AVG(reopen_count), 2) AS avg_reopens
FROM incidents;
""", 'sql_kpi_summary.csv')

display(export_monthly.head())
display(export_priority.head())
display(export_groups.head())
display(export_top_long.head())
display(export_kpi_summary)
display(summary_export)

,opened_month,incidents,sla_rate_pct
0,2016-02,207,64.73
1,2016-03,8995,39.88
2,2016-04,7934,76.70
3,2016-05,7508,77.56
4,2016-06,5,80.00


,priority_text,incidents,sla_rate_pct,avg_resolution_hours
0,Moderate,23466,64.55,174.35
1,Low,774,84.11,283.19
2,High,408,0.74,152.46
3,Critical,270,2.22,266.08


,assignment_group,incidents,sla_rate_pct,avg_resolution_hours
0,Group 70,9444,83.84,50.22
1,Group 25,1243,42.80,251.94
2,Group 39,1199,68.81,71.96
3,Group 24,1060,64.81,136.59
4,Group 23,811,58.69,245.22


,number,assignment_group,category,priority_text,resolution_time_hours
0,INC0001839,Group 31,Category 45,Moderate,8070.17
1,INC0001349,Group 67,Category 45,Low,8020.27
2,INC0007343,Group 31,Category 46,Moderate,7803.68
3,INC0001881,Group 35,Category 55,Low,7316.45
4,INC0001978,Group 35,Category 55,Low,7313.93


,total_incidents,sla_rate_pct,avg_resolution_hours,avg_reassignments,avg_reopens
0,24918,63.44,178.17,0.94,0.01


,exported_tables,baseline_sla_rate_pct,baseline_resolution_hours
0,24918,63.44,178.17


## Power BI Integration

The exported SQL datasets are imported into Power BI to build:
- Executive Dashboard
- SLA Monitoring
- Assignment Group Performance
- Monthly Incident Trends
- Resolution Time Analysis

This approach separates data preparation (Python + SQL) from reporting (Power BI), following common BI project practices.

## Key SQL Deliverables

This notebook produces reusable analytical datasets including:
- KPI summary
- Monthly incident trends
- Assignment group performance
- Priority SLA analysis
- Longest resolution cases

These outputs are consumed directly by the Power BI dashboard.

## Conclusion

The SQL layer transforms the cleaned incident dataset into reusable business tables suitable for operational reporting.

The extracted KPIs highlight:
- SLA compliance trends
- Assignment group performance
- Resolution bottlenecks
- Operational workload distribution

These SQL outputs serve as the analytical foundation for the Power BI dashboard presented in the final notebook.

In [14]:
conn.close()